# DNB — Offline Smoke Tests

Interactive validation of the pipeline:

```
WaveletConvolution → TargetWaveDetector (activation) → AmplitudeMonitor (inhibition) → StimTrigger
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger
from dnb.validation.ground_truth import validate, Annotation
from test_data import TestData

import dnb
print(f'DNB v{dnb.__version__}')

td = TestData()  # temp dir, auto-cleaned
# td = TestData('test_output')  # persistent dir if you want to inspect files

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

---
## 1. Clean sine wave — verify phase detection

In [ ]:
path = td.clean_sine()
TARGET_PHASE = 0

pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 4.0),
            target_phase=TARGET_PHASE,
            phase_tolerance=0.1,
            amp_min=1000.0, amp_max=100000.0,
            warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id=None,
            backoff_s=0.0,
        ),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.05),
)

events = pipeline.run_offline()
stim1 = [e for e in events if e.event_type == EventType.STIM1]

print(f'{len(stim1)} STIM1 events')
if stim1:
    phases = np.array([e.metadata['phase'] for e in stim1])
    times = np.array([e.timestamp for e in stim1])
    print(f'Phase: mean={np.mean(phases):.3f} std={np.std(phases):.3f} (target={TARGET_PHASE:.3f})')

    # Load signal back for plotting
    data = np.load(str(path))
    sig = data['continuous'][0]
    t = np.arange(len(sig)) / 1000.0

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(t, sig, 'b-', lw=0.5, alpha=0.7)
    for e in stim1:
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1)
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].set_title('Clean 1 Hz sine + STIM1 markers')

    axes[1].scatter(times, phases, c='red', s=20, zorder=5)
    axes[1].axhline(TARGET_PHASE, color='green', ls='--', label=f'target={TARGET_PHASE:.2f}')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

---
## 2. Synthetic slow waves with noise

In [ ]:
path_sw, gt_events = td.slow_waves(snr=5.0)
sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Planted: {len(sw_gt)} slow waves')

# Load for plotting
data = np.load(str(path_sw))
sig = data['continuous'][0]
t = np.arange(len(sig)) / 1000.0

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.6)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.5, lw=1.5, label='planted SW' if e == sw_gt[0] else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Synthetic recording (SNR=5.0)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pipeline = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10),
        TargetWaveDetector(
            id='slow_wave', freq_range=(0.5, 4.0),
            target_phase=0, phase_tolerance=0.1,
            amp_min=1000.0, amp_max=10000.0, warmup_chunks=3,
        ),
        StimTrigger(
            activation_detector_id='slow_wave',
            inhibition_detector_id=None,
            backoff_s=2.0,
        ),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.05),
)

detections = pipeline.run_offline()
stim1 = [e for e in detections if e.event_type == EventType.STIM1]
print(f'STIM1: {len(stim1)}')

annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(stim1, annotations, time_tolerance=0.5, target_type='SW')
print(report.summary())

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2, label='GT' if e == sw_gt[0] else '')
for e in stim1:
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--', label='STIM1' if e == stim1[0] else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Detections (red) vs Ground Truth (green)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. IED inhibition

In [ ]:
path_ied, gt_ied = td.slow_waves_with_ieds(snr=5.0)
sw_gt_ied = [e for e in gt_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

data_ied = np.load(str(path_ied))
sig_ied = data_ied['continuous'][0]
t_ied = np.arange(len(sig_ied)) / 1000.0

# WITH inhibition
pipe_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10),
        TargetWaveDetector(id='slow_wave', freq_range=(0.5, 4.0), target_phase=0, phase_tolerance=0.05, amp_min=1000.0, amp_max=10000.0, warmup_chunks=3),
        AmplitudeMonitor(id='ied_monitor', freq_range=(80.0, 120.0), ref_freq_range=(0.5, 4.0), ratio_max=1.0, warmup_chunks=3),
        StimTrigger(activation_detector_id='slow_wave', inhibition_detector_id='ied_monitor', backoff_s=20.0, inhibition_cooldown_s=20.0),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.05),
)
stim1_inh = [e for e in pipe_inh.run_offline() if e.event_type == EventType.STIM1]

# WITHOUT inhibition
pipe_no = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10),
        TargetWaveDetector(id='slow_wave', freq_range=(0.5, 4.0), target_phase=0, phase_tolerance=0.05, amp_min=1000.0, amp_max=10000.0, warmup_chunks=3),
        StimTrigger(activation_detector_id='slow_wave', inhibition_detector_id=None, backoff_s=3.0),
    ],
    config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.05),
)
stim1_no = [e for e in pipe_no.run_offline() if e.event_type == EventType.STIM1]

def near_ieds(stims, ieds, win=1.0):
    return sum(1 for s in stims if any(abs(s.timestamp - i.timestamp) < win for i in ieds))

print(f'With inhibition:    {len(stim1_inh)} STIM1, {near_ieds(stim1_inh, ied_gt)} near IEDs')
print(f'Without inhibition: {len(stim1_no)} STIM1, {near_ieds(stim1_no, ied_gt)} near IEDs')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, stims, label in [(axes[0], stim1_no, 'WITHOUT inhibition'), (axes[1], stim1_inh, 'WITH inhibition')]:
    ax.plot(t_ied, sig_ied, 'b-', lw=0.3, alpha=0.5)
    for e in sw_gt_ied:
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2)
    for e in ied_gt:
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3)
    for e in stims:
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--')
    ax.set_ylabel('Amp')
    ax.set_title(label)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

---
## 4. SNR sweep

In [ ]:
sweep = td.snr_sweep()
results = []

for actual_snr, path, gt in sweep:
    pipe = Pipeline(
        source=FileSource(str(path)),
        modules=[
            WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10),
            TargetWaveDetector(id='slow_wave', freq_range=(0.5, 2.0), target_phase=pi, phase_tolerance=0.05, amp_min=100.0, warmup_chunks=3),
            StimTrigger(activation_detector_id='slow_wave', inhibition_detector_id=None, backoff_s=3.0),
        ],
        config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
    )
    dets = pipe.run_offline()
    s1 = [e for e in dets if e.event_type == EventType.STIM1]
    sw = [e for e in gt if e.metadata.get('type') == 'SW']
    anns = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw]
    r = validate(s1, anns, time_tolerance=0.5)
    results.append({'snr': actual_snr, **r.metrics, 'n_detected': len(s1)})
    print(f'SNR={actual_snr:.1f}: P={r.metrics["precision"]:.2f} R={r.metrics["recall"]:.2f} F1={r.metrics["f1"]:.2f}')

In [ ]:
snrs = [r['snr'] for r in results]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(snrs, [r['precision'] for r in results], 'o-', label='Precision', lw=2)
axes[0].plot(snrs, [r['recall'] for r in results], 's-', label='Recall', lw=2)
axes[0].plot(snrs, [r['f1'] for r in results], '^-', label='F1', lw=2)
axes[0].set_xlabel('SNR')
axes[0].set_ylabel('Score')
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title('Detection Performance vs SNR')
axes[0].legend()
axes[0].grid(alpha=0.3)

x = np.arange(len(snrs))
w = 0.25
axes[1].bar(x - w, [r['true_positives'] for r in results], w, label='TP')
axes[1].bar(x, [r['false_positives'] for r in results], w, label='FP')
axes[1].bar(x + w, [r['false_negatives'] for r in results], w, label='FN')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s:.0f}' for s in snrs])
axes[1].set_xlabel('SNR')
axes[1].set_ylabel('Count')
axes[1].set_title('Detection Breakdown')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 5. Inspect wavelet output

In [ ]:
from dnb.modules.base import ProcessResult
from dnb.core.ring_buffer import RingBuffer
from dnb.core.types import DataChunk

path_short, gt_short, sig_short = td.short_segment()

config = PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.05)
wavelet_mod = WaveletConvolution(freq_min=0.5, freq_max=4, n_freqs=10)
wavelet_mod.configure(config)

ring = RingBuffer(n_channels=1, capacity=config.buffer_samples)
chunk_data = sig_short[:, :config.chunk_samples]
ring.write(chunk_data)

chunk = DataChunk(
    samples=chunk_data,
    timestamps=np.arange(config.chunk_samples) / 1000.0,
    channel_ids=np.array([0], dtype=np.int32),
    sample_rate=1000.0,
)
result = ProcessResult(chunk=chunk, ring_buffer=ring)
result = wavelet_mod.process(result)

wav = result.wavelet
t_chunk = chunk.timestamps

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(t_chunk, chunk.samples[0], 'b-', lw=0.5)
axes[0].set_ylabel('Raw signal')
axes[0].set_title('Wavelet decomposition of first chunk')

im = axes[1].pcolormesh(t_chunk, wav.frequencies, wav.amplitude[0], shading='auto', cmap='hot')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_yscale('log')
plt.colorbar(im, ax=axes[1], label='Amplitude')

so_mask = (wav.frequencies >= 0.5) & (wav.frequencies <= 2.0)
so_phase = np.mean(wav.phase[0, so_mask, :], axis=0)
axes[2].plot(t_chunk, so_phase, 'g-', lw=0.5)
axes[2].axhline(pi, color='red', ls='--', alpha=0.5, label='target phase (π)')
axes[2].set_ylabel('SO Phase (rad)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 6. Parameter exploration

In [ ]:
path_pe, gt_pe = td.slow_waves(snr=5.0, seed=99)
sw_pe = [e for e in gt_pe if e.metadata.get('type') == 'SW']
anns_pe = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_pe]

tolerances = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
tol_results = []

for tol in tolerances:
    pipe = Pipeline(
        source=FileSource(str(path_pe)),
        modules=[
            WaveletConvolution(freq_min=0.5, freq_max=30, n_freqs=10),
            TargetWaveDetector(id='slow_wave', freq_range=(0.5, 2.0), target_phase=pi, phase_tolerance=tol, amp_min=50.0, warmup_chunks=3),
            StimTrigger(activation_detector_id='slow_wave', inhibition_detector_id=None, backoff_s=3.0),
        ],
        config=PipelineConfig(sample_rate=1000, n_channels=1, chunk_duration=0.5),
    )
    dets = pipe.run_offline()
    s1 = [e for e in dets if e.event_type == EventType.STIM1]
    r = validate(s1, anns_pe, time_tolerance=0.5)
    tol_results.append({'tolerance': tol, **r.metrics, 'n_detected': len(s1)})
    print(f'tol={tol:.2f}: {len(s1)} STIM1, P={r.metrics["precision"]:.2f} R={r.metrics["recall"]:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tolerances, [r['precision'] for r in tol_results], 'o-', label='Precision')
ax.plot(tolerances, [r['recall'] for r in tol_results], 's-', label='Recall')
ax.plot(tolerances, [r['f1'] for r in tol_results], '^-', label='F1')
ax.set_xlabel('Phase tolerance (rad)')
ax.set_ylabel('Score')
ax.set_title('Effect of phase tolerance on detection')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()